# Meaning is geometry — embeddings & cosine similarity

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 8 §8.3 · type: concept-lab

**The promise:** by the end you can see that semantically similar texts land *near each other* in vector space — measured by cosine similarity — and explain why that one trick powers search-by-meaning.

Runs free and offline by default: `MOCK=1` ships small canned embedding vectors so the cosine math is deterministic with no API key.

## 🧠 Why this matters

Inside a model, every piece of text is a vector — a long list of numbers. The deepest idea in modern AI is that **meaning becomes geometry**: things that *mean* similar things end up *near* each other in that space. Nobody hand-defined "similar"; it emerged, because text used in similar contexts got pushed toward similar vectors by the prediction objective.

**Embedding models** are trained specifically to map a whole text — a sentence, a paragraph, a document — to a single vector so that related texts land close together. Closeness is measured by *cosine similarity*: near 1 means very similar, near 0 means unrelated. This is the engine under retrieval (Chapter 13), long-term memory (Chapter 14), dedup, clustering, and recommendation. See §8.3.

## Objectives & prereqs

**By the end you can:**
- Implement `cosine(a, b)` from scratch and sanity-check it on hand-made vectors.
- Embed short texts and read a cosine score with intuition.
- Rank sentences against a query by meaning, and see the win over keyword overlap.
- Explain why a vector database is "just an index for nearest vectors."

**Prereqs:** [`08-01-tokens-and-the-bill.ipynb`](08-01-tokens-and-the-bill.ipynb) (text → tokens). Chapter 8 read.

**Packages:** standard library only in `MOCK=1`. `MOCK=0` uses `openai` (in `requirements.txt`) to call a real embeddings endpoint.

In [ ]:
# Setup — imports, env, the MOCK switch, and a fixed seed.
import os
import random

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): canned embedding vectors -> deterministic, free, offline.
# MOCK=0: real embeddings endpoint (about a few cents of tokens for this notebook).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(8)  # determinism for anything stochastic

print(f"MOCK mode: {MOCK}  (canned vectors, offline)" if MOCK else "LIVE mode: calling a real embeddings endpoint")

## Cosine similarity, from scratch

Cosine similarity is the cosine of the angle between two vectors: their dot product divided by the product of their lengths. No library needed — it's a few lines, and writing it yourself demystifies every vector database you'll ever touch.

In [ ]:
def cosine(a: list[float], b: list[float]) -> float:
    """Cosine similarity of two equal-length vectors: dot(a, b) / (|a| * |b|).

    Returns a value in [-1, 1]; for embedding vectors it's typically in [0, 1],
    where ~1 means 'very similar meaning' and ~0 means 'unrelated'.
    """
    dot = sum(x * y for x, y in zip(a, b))
    norm = (sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5)
    return dot / norm if norm else 0.0


# Sanity checks on hand-made vectors, where we know the right answer by inspection:
print("identical   :", round(cosine([1, 0, 0], [1, 0, 0]), 3))   # -> 1.0
print("orthogonal  :", round(cosine([1, 0, 0], [0, 1, 0]), 3))   # -> 0.0
print("same dir x2 :", round(cosine([1, 1, 0], [2, 2, 0]), 3))   # -> 1.0 (length doesn't matter)
print("opposite    :", round(cosine([1, 0, 0], [-1, 0, 0]), 3))  # -> -1.0

**What you just saw.** Cosine measures *direction*, not magnitude: `[1,1,0]` and `[2,2,0]` point the same way, so they score 1.0. That's exactly what we want for meaning — a long document and a short query about the same topic should be "close" regardless of length.

## An embedding function (mock or live)

Now we need vectors that actually encode *meaning*. A real embedding model gives us those; for offline determinism, `MOCK=1` ships a tiny hand-built lookup whose vectors are arranged so related sentences sit close together — enough to feel the geometry without a network call.

The live path mirrors the book's §8.3 code shape: one `embed(texts)` call returning one vector per text.

In [ ]:
# Canned vectors for MOCK mode. Hand-placed in a small space so that texts about
# 'account access' cluster, and the unrelated weather sentence sits apart. These
# are illustrative, NOT real model outputs — the point is the geometry, not the
# numbers. Keys are normalized (lowercased) lookups.
_CANNED = {
    "how do i reset my password?":      [0.90, 0.30, 0.10, 0.05],
    "i can't log in to my account":     [0.85, 0.38, 0.12, 0.04],
    "i forgot my login credentials":    [0.88, 0.33, 0.09, 0.06],
    "how do i change my billing card?": [0.40, 0.20, 0.80, 0.10],
    "what's the weather like today?":   [0.05, 0.10, 0.08, 0.95],
    "the cat sat on the warm windowsill": [0.08, 0.92, 0.10, 0.20],
}


def embed(texts: list[str]) -> list[list[float]]:
    """Map each text to one embedding vector.

    MOCK=1: deterministic canned vectors (offline, free). Unknown strings get a
    stable pseudo-vector so the function never crashes on your own inputs.
    MOCK=0: a real embeddings endpoint (the book uses this exact shape in §8.3).
    """
    if MOCK:
        out = []
        for t in texts:
            key = t.strip().lower()
            if key in _CANNED:
                out.append(list(_CANNED[key]))
            else:
                # Stable fallback: seed a tiny RNG from the text so repeated runs match.
                rng = random.Random(key)
                out.append([rng.random() for _ in range(4)])
        return out

    from openai import OpenAI  # imported lazily so MOCK users need no key

    client = OpenAI()  # reads OPENAI_API_KEY from the environment
    resp = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in resp.data]


v = embed(["how do i reset my password?"])[0]
print(f"one vector, dim = {len(v)} (canned)" if MOCK else f"one vector, dim = {len(v)} (live)")
print("first few components:", [round(x, 3) for x in v[:4]])

## 🔮 Predict: which sentence is closer?

Query: **"How do I reset my password?"**

Candidate A: **"I can't log in to my account"** — same intent, *zero shared keywords*.
Candidate B: **"What's the weather like today?"** — unrelated.

Predict the two cosine scores before running. Which candidate wins, and by how much? (This is the book's §8.3 example pair.)

In [ ]:
query = "How do I reset my password?"
candidate_a = "I can't log in to my account"
candidate_b = "What's the weather like today?"

vq, va, vb = embed([query, candidate_a, candidate_b])
print(f"query: {query!r}\n")
print(f"  vs A {candidate_a!r}")
print(f"     cosine = {cosine(vq, va):.3f}   (high: same meaning, no shared keywords)")
print(f"  vs B {candidate_b!r}")
print(f"     cosine = {cosine(vq, vb):.3f}   (low: unrelated)")

**What you just saw.** "Reset my password" and "can't log in" share *no words*, yet they're close in vector space, while the weather sentence is far. Keyword search would miss the match entirely; meaning-search finds it. That gap is the whole reason retrieval is geometric.

## Ranking a small set by meaning

Embed a handful of sentences, score each against one query by cosine, and sort. This is a one-cell preview of what a vector database does at scale: store a vector per item, then return the nearest ones to a query vector.

In [ ]:
corpus = [
    "I can't log in to my account",
    "I forgot my login credentials",
    "How do I change my billing card?",
    "What's the weather like today?",
    "The cat sat on the warm windowsill",
]
query = "How do I reset my password?"

qv = embed([query])[0]
doc_vectors = embed(corpus)

ranked = sorted(
    ((cosine(qv, dv), text) for dv, text in zip(doc_vectors, corpus)),
    reverse=True,
)

print(f"query: {query!r}\n")
print("ranked by cosine (search by meaning):")
for score, text in ranked:
    print(f"  {score:.3f}  {text}")

# Contrast: naive keyword overlap (shared words) would rank these very differently.
def keyword_overlap(a: str, b: str) -> int:
    return len(set(a.lower().split()) & set(b.lower().split()))

print("\nshared-keyword count with the query (the old way):")
for text in corpus:
    print(f"  {keyword_overlap(query, text)}  {text}")

**What you just saw.** The top results by cosine are the account-access sentences — even the one with no shared words. By raw keyword overlap, the best matches are near-zero, because natural language rarely repeats the query's exact words. Meaning-search is what makes "find the relevant chunk" actually work.

## Foreshadow: RAG and memory

Hold the geometric picture and the rest of the book stops being exotic:

- **RAG (Chapter 13):** every document chunk becomes a vector; a user query becomes a vector; retrieval is "find the chunks whose vectors are nearest the query." A **vector database is just an index optimized for that nearest-neighbor lookup** — same `cosine` idea, made fast over millions of vectors.
- **Memory (Chapter 14):** past interactions get embedded and recalled by proximity to the current context.

Everything is the cell you just ran, scaled up.

## ⚠️ Pitfall: mixing models or skipping normalization

- **Never compare vectors from *different* embedding models.** Each model has its own geometry; a cosine score between a `model-A` vector and a `model-B` vector is meaningless. Re-embed your whole corpus when you change embedding models.
- **Mind normalization.** Cosine handles length on its own (it divides by the norms), but some pipelines pre-normalize and then use a plain dot product. Mixing normalized and un-normalized vectors, or forgetting which your store uses, silently corrupts ranking. Pick one convention and keep the corpus consistent.

## 🎯 Senior lens

A senior treats the embedding model as a versioned dependency, exactly like the LLM. The corpus, the query path, and the stored vectors must all use the *same* model and the *same* normalization, or retrieval quietly degrades. So the embedding call goes behind one interface (just like the LLM client in Chapter 11), the model id and dimension are pinned and recorded alongside the index, and a re-embed is a planned migration — not something you do to half the corpus on a Tuesday. The cheapest retrieval bug to avoid is the one where someone swapped the embedding model and never rebuilt the index.

## Recap

- Embeddings turn text into **vectors**; similar meanings land **near each other**.
- **Cosine similarity** measures direction (meaning), ignoring length; ~1 = similar, ~0 = unrelated.
- Meaning-search beats keyword overlap because related text rarely repeats the same words.
- A **vector database is an index for nearest vectors** — RAG and memory are this idea at scale.
- Don't compare across embedding models, and keep normalization consistent.

## Exercises

Each task *changes* something; predict the outcome before running.

1. **Add a paraphrase.** Add "I'm locked out and need access" to the corpus. Predict where it ranks for the password query, then check. Did meaning-search catch it despite new vocabulary?
2. **Break it on purpose.** Replace one corpus vector with a copy of an unrelated one (in the canned dict) and re-rank. Watch a wrong item jump to the top — a concrete picture of "stale/duplicated vectors corrupt retrieval."
3. **Top-k helper.** Write `top_k(query, corpus, k=2)` returning the `k` highest-cosine items. Predict the top 2 for a billing query before running.
4. **(Live, optional)** Set `COMPANION_MOCK=0` with a key and re-run the ranking on real embeddings. Are the relative orderings the same as the canned geometry?

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Next notebook:** [`08-03-context-and-lost-in-the-middle.ipynb`](08-03-context-and-lost-in-the-middle.ipynb) — vectors flow through *attention*; see why long context costs more and why a fact's *position* changes whether the model uses it.
- **Book:** §8.3 (embeddings); look ahead to §8.1 (attention) and §8.5 (context windows).
- **Where this leads:** the embeddings intuition here is the backbone of the RAG pipeline ([`blueprints/rag-pipeline/`](../../blueprints/rag-pipeline/)) and the capstone's `rag/` module (Chapter 13). This chapter doesn't build it — it makes you *trust* it.